# Agente de IA para cotizar notebooks en SoloTodo usando LangChain

Requerimientos:

- Crear un archivo .env con los campos `OPENAI_API_KEY` y `TAVILY_API_KEY`

In [9]:
%pip install jupyterlab langchain langchain-openai langchain-tavily langgraph-checkpoint-sqlite requests python-dotenv > /dev/null

Note: you may need to restart the kernel to use updated packages.


In [10]:
from langchain.chat_models import init_chat_model

In [ ]:
import os
import re
from uuid import uuid4

import requests
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import Command, interrupt
from pydantic import BaseModel, Field

load_dotenv(".env")

CLP_PER_USD = Decimal("994.58")
CHECKPOINT_DB = "solotodo-checkpoints.sqlite"


def all_rows(base, path, params=None):
    url = urljoin(base, path)
    while url:
        response = requests.get(url, params=params, timeout=20)
        response.raise_for_status()
        page = response.json()
        if isinstance(page, list):
            yield from page
            break
        yield from page["results"]
        next_url = page.get("next")
        url = urljoin(url, next_url) if next_url else None
        params = None

## 1. Define the demo catalogs

Fill in the SoloTodo IDs and speed scores you want to use in the demo. The example
entries use `None` for values that still need to be filled in. These dictionaries
are the only source for brand and component lookups at runtime.

For a direct demo without filling them in, try the price-and-RAM example in section 5.

In [ ]:
BRANDS = {
    "HP": None,
    "Lenovo": None,
    "ASUS": None,
}

PROCESSORS = {
    "Intel Core i5-1335U": {"id": None, "speed_score": None},
    "AMD Ryzen 7 7730U": {"id": None, "speed_score": None},
}

VIDEO_CARDS = {
    "NVIDIA GeForce RTX 4050 Laptop GPU": {"id": None, "speed_score": None},
    "NVIDIA GeForce RTX 4060 Laptop GPU": {"id": None, "speed_score": None},
}


@tool
def notebook_processors() -> list[dict]:
    response = requests.get('https://api.solotodo.com/notebook_processors/')
    return [{
        "id": x['id'],
        "name": x['name'],
        "performance_score": x['speed_score']
    } for x in response.json()]
    """Look up CPUs in the demo dictionary. Null means the value is not configured."""

@tool
def notebook_video_cards(name_contains: str = "") -> list[dict]:
    """Look up GPUs in the demo dictionary. Null means the value is not configured."""
    return [
        {"name": name, **details}
        for name, details in VIDEO_CARDS.items()
        if name_contains.casefold() in name.casefold()
    ]

## 2. Search the SoloTodo catalog

The tool accepts all budgets in Chilean pesos. It divides the maximum price by
`994.58`, rounds down to USD cents, and sends `offer_price_usd_max` to SoloTodo.
Brand IDs come from `BRANDS` and processor IDs come from the CPU tool. RAM choice
IDs still come from SoloTodo's filter layout.

In [ ]:
def objects(value):
    """Walk the nested layout objects, including fields grouped into sections."""
    if isinstance(value, dict):
        yield value
        for child in value.values():
            yield from objects(child)
    elif isinstance(value, list):
        for child in value:
            yield from objects(child)


@lru_cache(maxsize=1)
def filter_layout(base):
    return list(all_rows(base, "category_specs_form_layouts/", {"category": 1, "website": 1}))


def choices(base, field_name):
    for field in objects(filter_layout(base)):
        if any(field.get(key) == field_name for key in ("name", "key", "field", "field_name", "slug")):
            for key in ("choices", "options"):
                if field.get(key):
                    return field[key]
        nested = field.get(field_name)
        if isinstance(nested, dict):
            for key in ("choices", "options"):
                if nested.get(key):
                    return nested[key]
    return []


def choice_pairs(base, field_name):
    for choice in choices(base, field_name):
        if isinstance(choice, dict):
            value = choice.get("value", choice.get("id"))
            label = choice.get("label") or choice.get("name") or choice.get("display_name")
        else:
            value, label = choice
        if label is not None:
            yield object_id(value), str(label)


def ram_choice(base, requested_gb):
    thresholds = []
    for value, label in choice_pairs(base, "ram_quantity_min"):
        match = re.fullmatch(r"\s*(\d+(?:[.,]\d+)?)\s*(GB|GiB|MB|MiB)?\s*", label, re.I)
        if match:
            size = Decimal(match[1].replace(",", "."))
            if (match[2] or "GB").upper() in {"MB", "MIB"}:
                size /= 1024
            if size >= requested_gb:
                thresholds.append((size, value))
    return min(thresholds)[1]


def normalize_product(row):
    product = row.get("product", row.get("entity", row))
    picture = product.get("picture_url") or product.get("picture") or row.get("picture_url")
    if isinstance(picture, dict):
        picture = picture.get("url")
    return {
        "product_id": object_id(product),
        "product_name": product["name"],
        "picture_url": urljoin(PUBLIC_BASE, picture) if picture else None,
        "description": product.get("description") or row.get("description"),
        "specs": product.get("specs") or row.get("specs"),
        "prices_per_currency": row.get("prices_per_currency") or product.get("prices_per_currency"),
        "offer_price_usd": row.get("offer_price_usd", product.get("offer_price_usd")),
    }



@tool
def search_notebooks(
    brands: list[str] | None = None,
    processors: list[int] | None = None,
    max_price_clp: int | None = None,
    min_ram_gb: int | None = None,
    limit: int = 12,
) -> dict:
    """Find notebooks by brand name, SoloTodo CPU ID, maximum price in CLP, and minimum RAM in GB."""
    params = [("ordering", "offer_price_usd")]
    brand_ids = {name.casefold(): value for name, value in BRANDS.items()}
    for brand in brands or []:
        brand_id = brand_ids.get(brand.strip().casefold())
        if brand_id is None:
            return {"error": f"No SoloTodo brand ID configured for {brand}."}
        params.append(("brands", brand_id))
    params.extend(("processors", cpu_id) for cpu_id in processors or [])
    if max_price_clp is not None:
        usd = (Decimal(max_price_clp) / CLP_PER_USD).quantize(
            Decimal("0.01"), rounding=ROUND_DOWN
        )
        params.append(("offer_price_usd_max", str(usd)))
    if min_ram_gb is not None:
        params.append(("ram_quantity_min", ram_choice(PUBLIC_BASE, min_ram_gb)))

    rows = all_rows(PUBLIC_BASE, NOTEBOOK_BROWSE, params)
    return {"results": [normalize_product(row) for row in islice(rows, limit)]}

## 3. Define the agent

`TavilySearch` is available alongside the SoloTodo tools. The `ask_user` tool
calls LangGraph's `interrupt()`. The agent returns its Pydantic result under
`structured_response` when the run finishes.

In [ ]:
class Laptop(BaseModel):
    product_id: int
    product_name: str
    picture_url: str | None
    description: str = Field(description="Explain why this laptop fits the request.")


class Recommendations(BaseModel):
    summary: str
    products: list[Laptop] = Field(max_length=3)


@tool
def ask_user(question: str) -> str:
    """Pause the run to ask one question that would materially improve the recommendations."""
    return interrupt({"question": question})


SYSTEM_PROMPT = """You help people choose a notebook in Chile using SoloTodo.
Respond in the shopper's language. For a vague request, call ask_user with one
focused question about missing budget or intended use. Do this before browsing.
If there is enough information to make a useful shortlist, proceed directly.
Interpret every budget as CLP. Apply explicit filters. If a requested brand or
CPU ID is unconfigured, report it and keep that requirement.
Resolve named CPUs with notebook_processors. For performance-sensitive requests,
use the CPU/GPU catalogs to interpret hardware and Tavily for recent benchmark or
review context when it could change the ranking. GPU scores describe components,
and laptop cooling and GPU power limits can change real-world performance.
The browse tool supports CPU filters. Check visible GPU specs for each candidate
before claiming it matches a GPU requirement. Report if the candidate set is
insufficient to verify a match. Treat search snippets as reference material.
Recommend up to three actual products returned by search_notebooks. Copy the
product ID, name and picture URL from SoloTodo, using null for a missing image.
Explain each pick in its description. Include source URLs in the summary if
external benchmark claims affect the picks. If no verified matches are available,
return an empty products list and explain the reason.
"""


def build_agent(checkpointer):
    model = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4.1-mini"),
                       parallel_tool_calls=False, timeout=45)
    return create_agent(
        model=model,
        tools=[notebook_processors, notebook_video_cards, search_notebooks,
               TavilySearch(max_results=3), ask_user],
        system_prompt=SYSTEM_PROMPT,
        response_format=ToolStrategy(Recommendations),
        checkpointer=checkpointer,
    )

## 4. Pause and resume

Start with a vague request. The agent should call `ask_user`, checkpoint its state,
and return the question. The database connection closes at the end of the cell.

In [ ]:
thread_id = uuid4().hex
config = {"configurable": {"thread_id": thread_id}}

with SqliteSaver.from_conn_string(CHECKPOINT_DB) as saver:
    agent = build_agent(saver)
    result = agent.invoke(
        {"messages": [{"role": "user", "content": "Help me pick a laptop."}]},
        config=config,
    )

if result.get("__interrupt__"):
    print(result["__interrupt__"][0].value["question"])
    print("Thread ID:", thread_id)
else:
    print(result["structured_response"].model_dump_json(indent=2))

Edit the answer below. The cell opens the same SQLite file and creates a new agent
with the same tools. Passing the same thread ID tells LangGraph which checkpoint to
load. To try this after a kernel restart, run the setup cells and set `thread_id`
to the printed value before running the resume cell. If the agent asks another
question, edit the answer and run the resume cell again.

In [ ]:
answer = "I need it for university and Python development. My budget is CLP 600,000."
# thread_id = "paste the original thread ID here after restarting the kernel"
config = {"configurable": {"thread_id": thread_id}}

with SqliteSaver.from_conn_string(CHECKPOINT_DB) as saver:
    agent = build_agent(saver)
    result = agent.invoke(Command(resume=answer), config=config)

if result.get("__interrupt__"):
    print(result["__interrupt__"][0].value["question"])
else:
    print(result["structured_response"].model_dump_json(indent=2))

## 5. Try a direct catalog search

This request supplies enough detail for the agent to search immediately. For a
performance-oriented run, try asking for a gaming laptop under CLP 1,500,000 for
Cyberpunk 2077 at 1080p and ask it to check current benchmarks.

Once you have filled in the HP ID in `BRANDS`, add “HP” to the example to demonstrate brand filtering.

In [ ]:
config = {"configurable": {"thread_id": uuid4().hex}}
request = "Find a laptop with at least 16 GB RAM for up to CLP 650,000."

with SqliteSaver.from_conn_string(CHECKPOINT_DB) as saver:
    agent = build_agent(saver)
    result = agent.invoke({"messages": [{"role": "user", "content": request}]}, config=config)

if result.get("__interrupt__"):
    print(result["__interrupt__"][0].value["question"])
    print("Thread ID:", config["configurable"]["thread_id"])
else:
    print(result["structured_response"].model_dump_json(indent=2))